# Valence Band Offset of SiO₂/HfO₂ (001) Superlattices with Monoclinic and Cubic HfO₂

> **Pierre-Yves Prodhomme, Fabien Fontaine-Vive, Abram Van Der Geest, Philippe Blaise and Jacky Even**, "Ab initio
> calculation of effective work functions for a TiN/HfO2/SiO2/Si transistor stack", Applied Physics Letters 99,
> 022101 (2011). [DOI: 10.1063/1.3609869](https://doi.org/10.1063/1.3609869)

> **John Robertson and Robert M. Wallace**, "High-K materials and metal gates for CMOS applications", Materials
> Science and Engineering: R: Reports 88, 1 (2015). [DOI: 10.1016/j.mser.2014.11.001](https://doi.org/10.1016/j.mser.2014.11.001)

Computes the valence band offset of the two SiO₂/HfO₂ (001) superlattices created in the
[structure notebook](interface_3d_3d_silicon_dioxide_hafnium_dioxide.ipynb), with the platform's
[Valence Band Offset](../workflows/valence_band_offset.ipynb) workflow on each relaxed superlattice and its two bulk
cells, and compares each offset with its publication. Each bulk's valence band maximum is measured from its own average
electrostatic potential V̄, and the superlattice gives the step in V̄ between the two materials:

ΔE_v = (E_VBM − V̄)_SiO₂ − (E_VBM − V̄)_HfO₂ + (V̄_SiO₂ − V̄_HfO₂)_superlattice

ΔE_v is positive when the valence band maximum of SiO₂ lies above that of HfO₂, the sign convention of both
publications. Both publications use LDA, as this notebook does; the publications' values compared here are LDA
values, and their G0W0 corrections to the band edges are not computed.

- **Monoclinic HfO₂**, after Prodhomme et al.: the atomic positions are annealed from 1000 K to 100 K in 1 ps, then
  relaxed, as in the publication.
- **Cubic HfO₂**, the model of Robertson and Wallace, built after Sharia et al.: the atomic positions are relaxed.

Both publications relax with DFT in LDA, Prodhomme et al. with SIESTA and Sharia et al. with VASP; here the anneal and
the relaxation use the MACE-MP-0 machine-learned force field, trained on PBE data. Both move only the cation layer on
each side of an interface and the atoms between them; the block interiors are held at the bulk crystals.

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](interface_3d_3d_silicon_dioxide_hafnium_dioxide.ipynb), which saves the two superlattices and their three bulk cells to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.5, or use the defaults.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results; the last section prints the comparison with the manuscripts.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model, anneal, relaxation and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the two superlattices and the three bulks by name from the `uploads` folder or the platform, print their composition and geometry.
1. Configure the DFT model and a k-grid per material.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Anneal the monoclinic superlattice with molecular dynamics on the MACE force field.
1. Relax the atomic positions of both superlattices on the same force field.
1. Configure the Valence Band Offset workflow for each superlattice: the model, a k-grid per material and the band structures the valence band maxima are read from.
1. Run the jobs, or reuse ones that already ran on the same materials.
1. Retrieve the valence band maxima and the potential profiles, compute the offsets, and plot the potentials with the bulk potentials overlaid.
1. Compare with the manuscripts.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.calculators.mlff import get_mlff_install_profiles
from mat3ra.notebooks_utils.packages import install_packages

await install_packages(get_mlff_install_profiles("mace"))

from mat3ra.notebooks_utils.pyodide.packages.patches import apply_all_patches

apply_all_patches("mace")

### 1.2. Material names

In [ ]:
# Names saved by interface_3d_3d_silicon_dioxide_hafnium_dioxide.ipynb.
MONOCLINIC_SUPERLATTICE_NAME = "SiO2/HfO2 monoclinic (001) superlattice 10+7"
CUBIC_SUPERLATTICE_NAME = "SiO2/HfO2 cubic (001) superlattice 12+6"
SUBSTRATE_BULK_NAME = "SiO2 beta-cristobalite (001) bulk"
MONOCLINIC_FILM_BULK_NAME = "HfO2 monoclinic (001) bulk strained"
CUBIC_FILM_BULK_NAME = "HfO2 cubic (001) bulk strained"

FILM_BULK_NAME_BY_SUPERLATTICE_NAME = {
    MONOCLINIC_SUPERLATTICE_NAME: MONOCLINIC_FILM_BULK_NAME,
    CUBIC_SUPERLATTICE_NAME: CUBIC_FILM_BULK_NAME,
}
ANNEALED_SUPERLATTICE_NAMES = [MONOCLINIC_SUPERLATTICE_NAME]  # annealed before the relaxation, as Prodhomme et al.

### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

VBO_WORKFLOW_SEARCH_TERM = "valence_band_offset.json"
POTENTIAL_WORKFLOW_SEARCH_TERM = "average_electrostatic_potential.json"
MY_WORKFLOW_NAME = "Valence Band Offset"
APPLICATION_NAME = "espresso"

CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.OR
PPN = 16  # the OR queue allows at most 16 cores per node
TIME_LIMIT = "04:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.4. DFT model parameters

In [ ]:
MODEL_SUBTYPE = "lda"
FUNCTIONAL = "pz"  # both publications use LDA
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft, the LDA family the platform publishes for Hf, Si and O
VALENCE_ELECTRONS_PER_ATOM = {"Hf": 12, "Si": 4, "O": 6}  # GBRV LDA pseudopotentials, Hf with its 5s and 5p shells
BLOCK_LABEL = {SUBSTRATE_BULK_NAME: "SiO₂", MONOCLINIC_FILM_BULK_NAME: "monoclinic HfO₂", CUBIC_FILM_BULK_NAME: "cubic HfO₂"}
BLOCK_CATION = {SUBSTRATE_BULK_NAME: "Si", MONOCLINIC_FILM_BULK_NAME: "Hf", CUBIC_FILM_BULK_NAME: "Hf"}  # marks each block
CATIONS_PER_LAYER = {"Si": 1, "Hf": 2}  # per (001) layer of the 5.23 Å cell, one period of the planar-averaged potential
ECUTWFC = 40   # Ry, GBRV's recommended wavefunction cutoff
ECUTRHO = 200  # Ry, GBRV's recommended charge-density cutoff
ADDITIONAL_PARAMETERS = {
    # Hf, the third species of the superlattices: its GBRV atomic density holds 10 of its 12 valence electrons;
    # scaled by (12 + 2.4) / 12, the starting density is neutral in both blocks
    "system": {"starting_charge(3)": -2.4},
    "electrons": {"mixing_mode": "local-TF", "mixing_beta": 0.2, "mixing_ndim": 20, "electron_maxstep": 200},  # the long cells need local-TF mixing with a long history to converge
}
KPOINT_DENSITY = 3.5  # points per Å⁻¹, for the bulks
SUPERLATTICE_KGRID = [4, 4, 1]  # Sharia et al.

SCF_UNIT = "pw_scf"
BANDS_UNIT = "pw_bands"
AVERAGE_UNIT = "average ESP"

### 1.5. Anneal and relaxation parameters

In [ ]:
MACE_MODEL_FAMILY = "MACE-MP-0"
MACE_MODEL = "large"
MACE_DISPERSION = False
MACE_DEFAULT_DTYPE = "float32"
MACE_DEVICE = "cpu"

ANNEAL_START_TEMPERATURE = 1000  # K
ANNEAL_END_TEMPERATURE = 100  # K
ANNEAL_DURATION = 1000  # fs
ANNEAL_TIMESTEP = 1  # fs
ANNEAL_FRICTION = 0.01  # 1/fs, Langevin thermostat
ANNEAL_SEED = 42  # initial velocities and thermostat noise
ANNEAL_LOG_INTERVAL = 100  # steps
RELAXATION_FORCE_THRESHOLD = 0.07  # eV/Å, the force threshold of Prodhomme et al.
INTERFACE_CATION_LAYERS = 1  # on each side of an interface; these and the atoms between them move, all others are fixed

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load from the uploads folder and print composition and geometry

Both superlattices are periodic along [001], with two SiO₂/HfO₂ interfaces each; HfO₂ is strained in plane to the
5.23 Å cell of SiO₂, and each HfO₂ bulk cell carries the same strain.

In [ ]:
import numpy as np
from collections import Counter
from mat3ra.notebooks_utils.core.entity.material.api import load_material

MATERIAL_NAMES = [*FILM_BULK_NAME_BY_SUPERLATTICE_NAME, SUBSTRATE_BULK_NAME, *FILM_BULK_NAME_BY_SUPERLATTICE_NAME.values()]
materials_by_name = {name: load_material(client, FOLDER, name, ACCOUNT_ID) for name in MATERIAL_NAMES}


def get_layer_spacing(bulk, bulk_name):
    vectors = np.array(bulk.lattice.vector_arrays)
    normal_length = abs(np.linalg.det(vectors)) / np.linalg.norm(np.cross(vectors[0], vectors[1]))
    cation = BLOCK_CATION[bulk_name]
    return normal_length * CATIONS_PER_LAYER[cation] / bulk.basis.elements.values.count(cation)


valence_electrons = {}
for name, material in materials_by_name.items():
    composition = Counter(material.basis.elements.values)
    electron_count = sum(VALENCE_ELECTRONS_PER_ATOM[element] * count for element, count in composition.items())
    lattice = material.lattice
    valence_electrons[name] = electron_count
    print(f"{name}: {dict(composition)}, {material.basis.number_of_atoms} atoms, "
          f"cell {lattice.a:.4f} x {lattice.b:.4f} x {lattice.c:.4f} Å, {electron_count} valence electrons")
    if name in BLOCK_CATION:
        print(f"  (001) layer spacing: {get_layer_spacing(material, name):.4f} Å")

## 4. Configure the DFT model and k-grid
### 4.1. DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_SUBTYPE} {FUNCTIONAL}, {PSEUDOPOTENTIAL_TYPE} pseudopotentials")

### 4.2. k-grid per material

In [ ]:
from mat3ra.notebooks_utils.workflow import kgrid_from_density

kgrid = {name: kgrid_from_density(material, KPOINT_DENSITY) for name, material in materials_by_name.items()}
kgrid.update({name: SUPERLATTICE_KGRID for name in FILM_BULK_NAME_BY_SUPERLATTICE_NAME})
for name, grid in kgrid.items():
    print(f"{name}: k-grid {grid}")

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")

## 6. Anneal the monoclinic superlattice

Molecular dynamics at fixed cell: velocities are drawn at the start temperature, and the thermostat temperature is
lowered at every step, linearly to the end temperature. The annealed structure is the starting point of the relaxation.

In [ ]:
from ase import units
from ase.constraints import FixAtoms
from ase.md import MDLogger
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mat3ra.made.material import Material
from mat3ra.made.tools.convert import from_ase, to_ase
from mat3ra.notebooks_utils.calculators.mlff import create_mlff_calculator

calculator = create_mlff_calculator(
    "mace",
    {
        "family": MACE_MODEL_FAMILY,
        "model": MACE_MODEL,
        "dispersion": MACE_DISPERSION,
        "default_dtype": MACE_DEFAULT_DTYPE,
        "device": MACE_DEVICE,
    },
)
temperatures = np.linspace(ANNEAL_START_TEMPERATURE, ANNEAL_END_TEMPERATURE, round(ANNEAL_DURATION / ANNEAL_TIMESTEP))


def get_interior_indices(superlattice, film_cation, substrate_cation):
    atoms = to_ase(superlattice)
    heights, period = atoms.positions[:, 2], atoms.cell[2, 2]
    elements = np.array(atoms.get_chemical_symbols())
    film_center = np.angle(np.exp(2j * np.pi * heights[elements == film_cation] / period).mean()) * period / (2 * np.pi)
    offsets = (heights - film_center + period / 2) % period - period / 2
    in_interface = np.zeros(len(atoms), dtype=bool)
    for side_offsets in (offsets, -offsets):  # the interface above the film block, then the one below
        film_cations = np.sort(side_offsets[(elements == film_cation) & (side_offsets > 0)])
        substrate_cations = np.sort(side_offsets[(elements == substrate_cation) & (side_offsets > 0)])
        film_edge = film_cations[-INTERFACE_CATION_LAYERS * CATIONS_PER_LAYER[film_cation]]
        substrate_edge = substrate_cations[INTERFACE_CATION_LAYERS * CATIONS_PER_LAYER[substrate_cation] - 1]
        in_interface |= (side_offsets >= film_edge) & (side_offsets <= substrate_edge)
    return np.flatnonzero(~in_interface)


interior_indices = {
    name: get_interior_indices(materials_by_name[name], BLOCK_CATION[film_bulk_name], BLOCK_CATION[SUBSTRATE_BULK_NAME])
    for name, film_bulk_name in FILM_BULK_NAME_BY_SUPERLATTICE_NAME.items()
}

unrelaxed_superlattices = {name: materials_by_name[name] for name in FILM_BULK_NAME_BY_SUPERLATTICE_NAME}
for name in ANNEALED_SUPERLATTICE_NAMES:
    atoms = to_ase(materials_by_name[name])
    atoms.set_constraint(FixAtoms(indices=interior_indices[name]))
    atoms.calc = calculator
    random_generator = np.random.default_rng(ANNEAL_SEED)
    MaxwellBoltzmannDistribution(atoms, temperature_K=ANNEAL_START_TEMPERATURE, rng=random_generator)
    dynamics = Langevin(atoms, ANNEAL_TIMESTEP * units.fs, temperature_K=ANNEAL_START_TEMPERATURE,
                        friction=ANNEAL_FRICTION / units.fs, rng=random_generator)
    dynamics.attach(MDLogger(dynamics, atoms, "-", peratom=False), interval=ANNEAL_LOG_INTERVAL)
    for temperature in temperatures:
        dynamics.set_temperature(temperature_K=temperature)
        dynamics.run(1)
    annealed = Material.create(from_ase(atoms))
    annealed.name = f"{name} annealed"
    unrelaxed_superlattices[name] = annealed

## 7. Relax the superlattices

Fixed-cell relaxation of the atomic positions on the same force field, to the force threshold of Prodhomme et al.; the
monoclinic superlattice starts from its annealed structure. Each relaxed superlattice is named `<name> relaxed`.

In [ ]:
from ase.optimize import BFGS

relaxed_superlattices = {}
for name, superlattice in unrelaxed_superlattices.items():
    atoms = to_ase(superlattice)
    atoms.set_constraint(FixAtoms(indices=interior_indices[name]))
    atoms.calc = calculator
    BFGS(atoms).run(fmax=RELAXATION_FORCE_THRESHOLD)
    relaxed = Material.create(from_ase(atoms))
    relaxed.name = f"{name} relaxed"
    relaxed_superlattices[name] = relaxed

## 8. Configure the Valence Band Offset workflow

The workflow runs the band structure and electrostatic potential calculation on the two bulks, each with its own
k-grid, and stores the band structures. The superlattice needs only its electrostatic potential, so its band structure
calculation is replaced by the platform's Average Electrostatic Potential workflow. Section 10 computes the offset
from the band structures and the planar-averaged potentials, so the workflow's own lineup step is left out. One workflow is configured per
superlattice.

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.context.providers import PointsGridDataProvider
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, patch_workflow_qe_input

job_material_names, job_materials, workflows = {}, {}, {}
for name, film_bulk_name in FILM_BULK_NAME_BY_SUPERLATTICE_NAME.items():
    job_material_names[name] = [name, SUBSTRATE_BULK_NAME, film_bulk_name]
    job_materials[name] = [relaxed_superlattices[name], materials_by_name[SUBSTRATE_BULK_NAME],
                           materials_by_name[film_bulk_name]]
    workflow = Workflow.create(
        WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(VBO_WORKFLOW_SEARCH_TERM)
    )
    workflow.name = MY_WORKFLOW_NAME
    superlattice_band_structure = workflow.subworkflows[0]
    for subworkflow in list(workflow.subworkflows):
        if subworkflow.application.name != APPLICATION_NAME or subworkflow.id == superlattice_band_structure.id:
            workflow.remove_unit(workflow.get_unit_by_name(name=subworkflow.name).flowchartId)
            workflow.remove_subworkflow_by_id(subworkflow.id)
    potential_workflow = Workflow.create(
        WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(POTENTIAL_WORKFLOW_SEARCH_TERM)
    )
    workflow.add_subworkflow(potential_workflow.subworkflows[0], head=True)
    for unit_name in [SCF_UNIT, BANDS_UNIT]:
        apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=unit_name)
    for subworkflow in workflow.subworkflows:
        subworkflow.model = model
        material_index = int(next((unit.value for unit in subworkflow.units if unit.name.startswith("Set Material Index")), 0))
        material_name = job_material_names[name][material_index]

        if bands_unit := subworkflow.get_unit_by_name(name=BANDS_UNIT):
            bands_unit.results.append({"name": "band_structure"})
            subworkflow.set_unit(bands_unit)

        scf_unit = subworkflow.get_unit_by_name(name=SCF_UNIT)
        scf_unit.add_context(PointsGridDataProvider(material=job_materials[name][material_index],
                                                    dimensions=kgrid[material_name], isEdited=True).get_context_item_data())
        subworkflow.set_unit(scf_unit)

    patch_workflow_qe_input(workflow, ADDITIONAL_PARAMETERS, unit_names=[SCF_UNIT])
    workflows[name] = workflow
visualize_workflow(workflow)

## 9. Save the materials and run the jobs

For each superlattice, a job that already ran on the same three materials is reused; otherwise a new one is created
and submitted. One job runs at a time.

In [ ]:
from mat3ra.notebooks_utils.core.entity.job.api import get_kgrid_query
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

job_ids = {}
for name, workflow in workflows.items():
    saved_materials = [get_or_create_material(client, material, ACCOUNT_ID) for material in job_materials[name]]
    material_ids = [material["_id"] for material in saved_materials]
    candidates = client.jobs.list({
        "_material._id": material_ids[0], "owner._id": ACCOUNT_ID, "workflow.name": workflow.name,
        "status": {"$in": ["submitted", "queued", "active", "finished"]}, **get_kgrid_query(kgrid[name], SCF_UNIT),
    })
    job = next((candidate for candidate in candidates
                if [material["_id"] for material in candidate["_materials"]] == material_ids), None)
    if job is None:
        job = create_job(api_client=client, materials=saved_materials, workflow=workflow, project_id=project_id,
                         owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {name} {timestamp}")
        submit_jobs(client.jobs, [job["_id"]])
        print(f"✅ Submitted job {job['_id']}")
    else:
        print(f"♻️  Reusing job {job['_id']}")
    await wait_for_jobs_to_finish_async(client.jobs, [job["_id"]], poll_interval=POLL_INTERVAL)
    job_ids[name] = job["_id"]

## 10. Retrieve the results
### 10.1. Valence band maxima and average potentials

E_VBM is the top of the highest valence band, band N_electrons / 2, along each bulk's k-path, and the gap is the
bottom of the next band minus E_VBM. V̄ is the planar average of the electrostatic potential, averaged over the whole
cell for a bulk, and over one (001) layer, one period of the potential, at the centre of each block for a
superlattice, the bulk-like region far from both interfaces.

In [ ]:
band_structures, profiles = {}, {}
for name, job_id in job_ids.items():
    for subworkflow in client.jobs.get(job_id)["workflow"]["subworkflows"]:
        units = {unit["name"]: unit for unit in subworkflow["units"]}
        material_name = job_material_names[name][int(next((unit["value"] for unit in subworkflow["units"]
                                                            if unit["name"].startswith("Set Material Index")), 0))]
        profiles[name, material_name] = client.properties.get_for_job(
            job_id, property_name="average_potential_profile", unit_id=units[AVERAGE_UNIT]["flowchartId"])[0]
        if BANDS_UNIT in units:
            band_structures[name, material_name] = np.array(client.properties.get_for_job(
                job_id, property_name="band_structure", unit_id=units[BANDS_UNIT]["flowchartId"])[0]["yDataSeries"])

block_windows, superlattice_potential, bulk_potential, valence_band_maximum, bulk_gap = {}, {}, {}, {}, {}
for name, (_, *bulk_names) in job_material_names.items():
    superlattice = relaxed_superlattices[name]
    superlattice_in_cartesian = superlattice.clone()
    superlattice_in_cartesian.to_cartesian()
    heights = np.array(superlattice_in_cartesian.basis.coordinates.values)[:, 2]
    elements = np.array(superlattice.basis.elements.values)
    period = superlattice.lattice.c
    positions = np.array(profiles[name, name]["xDataArray"])
    planar_average = np.array(profiles[name, name]["yDataSeries"][0])
    print(f"{name}:")
    for bulk_name in bulk_names:
        key = (name, bulk_name)
        number_of_valence_bands = valence_electrons[bulk_name] // 2
        cation_heights = heights[elements == BLOCK_CATION[bulk_name]]
        # centre of the block's layers, also when the block wraps around the cell boundary
        center = np.angle(np.exp(2j * np.pi * cation_heights / period).mean()) * period / (2 * np.pi) % period
        spacing = get_layer_spacing(materials_by_name[bulk_name], bulk_name)
        block_windows[key] = (center, spacing)
        distances = (positions - center + period / 2) % period - period / 2
        superlattice_potential[key] = planar_average[np.abs(distances) <= spacing / 2].mean()
        bulk_potential[key] = np.mean(profiles[key]["yDataSeries"][0])
        valence_band_maximum[key] = band_structures[key][number_of_valence_bands - 1].max()
        bulk_gap[key] = band_structures[key][number_of_valence_bands].min() - valence_band_maximum[key]
        print(f"  {bulk_name}: E_VBM = {valence_band_maximum[key]:.4f} eV, V̄ = {bulk_potential[key]:.4f} eV, "
              f"E_VBM - V̄ = {valence_band_maximum[key] - bulk_potential[key]:.4f} eV, gap {bulk_gap[key]:+.3f} eV")
        print(f"    in the superlattice: V̄ = {superlattice_potential[key]:.4f} eV, averaged over "
              f"{spacing:.4f} Å centred at z = {center:.4f} Å")

### 10.2. Valence band offsets

In [ ]:
potential_lineup, valence_band_offset = {}, {}
for name, (_, substrate_bulk_name, film_bulk_name) in job_material_names.items():
    substrate, film = (name, substrate_bulk_name), (name, film_bulk_name)
    bulk_term = ((valence_band_maximum[substrate] - bulk_potential[substrate])
                 - (valence_band_maximum[film] - bulk_potential[film]))
    potential_lineup[name] = superlattice_potential[substrate] - superlattice_potential[film]
    valence_band_offset[name] = bulk_term + potential_lineup[name]
    film_label = BLOCK_LABEL[film_bulk_name]
    print(f"{name}:")
    print(f"  (E_VBM - V̄)_SiO₂ - (E_VBM - V̄)_{film_label} = {bulk_term:+.4f} eV")
    print(f"  (V̄_SiO₂ - V̄_{film_label})_superlattice = {potential_lineup[name]:+.4f} eV")
    print(f"  ΔE_v = E_VBM(SiO₂) - E_VBM({film_label}) = {valence_band_offset[name]:+.4f} eV")

### 10.3. Potential across the superlattices

One period from the HfO₂→SiO₂ interface, SiO₂ on the left. Each bulk's own potential (dotted) is overlaid on its
block, shifted to the block's average; where they coincide the block is bulk-like, and the two averages (dashed
horizontal lines) are shifted with respect to each other by the lineup.

In [ ]:
import matplotlib.pyplot as plt
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

figure, axes = plt.subplots(len(job_ids), 1, figsize=(10, 5 * len(job_ids)), squeeze=False)
for axis, (name, (_, substrate_bulk_name, film_bulk_name)) in zip(axes[:, 0], job_material_names.items()):
    superlattice = relaxed_superlattices[name]
    superlattice_in_cartesian = superlattice.clone()
    superlattice_in_cartesian.to_cartesian()
    heights = np.array(superlattice_in_cartesian.basis.coordinates.values)[:, 2]
    elements = np.array(superlattice.basis.elements.values)
    period = superlattice.lattice.c
    positions = np.array(profiles[name, name]["xDataArray"])
    planar_average = np.array(profiles[name, name]["yDataSeries"][0])

    substrate_center, substrate_spacing = block_windows[name, substrate_bulk_name]
    substrate_cation = BLOCK_CATION[substrate_bulk_name]
    substrate_layers = np.sum(elements == substrate_cation) / CATIONS_PER_LAYER[substrate_cation]
    plot_origin = substrate_center - substrate_layers * substrate_spacing / 2  # the middle of the HfO2->SiO2 interface
    order = np.argsort((positions - plot_origin) % period)
    axis.plot(((positions - plot_origin) % period)[order], planar_average[order], color="black", label="superlattice")
    for bulk_name, color in ((substrate_bulk_name, "tab:blue"), (film_bulk_name, "tab:red")):
        key = (name, bulk_name)
        bulk = materials_by_name[bulk_name]
        cation = BLOCK_CATION[bulk_name]
        center, spacing = block_windows[key]
        half_width = np.sum(elements == cation) / CATIONS_PER_LAYER[cation] * spacing / 2
        offsets = np.linspace(-half_width, half_width, 400)  # from the block's centre
        cation_offsets = (heights[elements == cation] - center + period / 2) % period - period / 2
        layer_offsets = np.sort(cation_offsets).reshape(-1, CATIONS_PER_LAYER[cation]).mean(axis=1)
        plane_offset = layer_offsets[np.abs(layer_offsets).argmin()]  # the layer nearest the block centre
        bulk_profile = profiles[key]
        profile_length = len(bulk_profile["xDataArray"]) * bulk_profile["xDataArray"][1]  # one bulk cell
        bulk_cation_fractions = np.array(bulk.basis.coordinates.values)[np.array(bulk.basis.elements.values) == cation][:, 2]
        bulk_layer_fraction = np.sort(bulk_cation_fractions)[:CATIONS_PER_LAYER[cation]].mean()
        bulk_normal_length = spacing * bulk.basis.elements.values.count(cation) / CATIONS_PER_LAYER[cation]
        bulk_values = np.interp((((offsets - plane_offset) / bulk_normal_length + bulk_layer_fraction) * profile_length),
                                bulk_profile["xDataArray"], bulk_profile["yDataSeries"][0], period=profile_length)
        region = offsets + (center - plot_origin) % period
        shift = superlattice_potential[key] - bulk_potential[key]
        axis.plot(region, bulk_values + shift, color=color, linestyle=":", linewidth=2,
                  label=f"bulk {BLOCK_LABEL[bulk_name]}, shifted")
        axis.hlines(superlattice_potential[key], region[0], region[-1], color=color, linestyle="--",
                    label=f"V̄_{BLOCK_LABEL[bulk_name]} = {superlattice_potential[key]:.3f} eV")
        axis.text((center - plot_origin) % period, 0.02, BLOCK_LABEL[bulk_name], transform=axis.get_xaxis_transform(),
                  ha="center")
    axis.set_xlim(0, period)
    axis.set_xlabel("position along (001) (Å)")
    axis.set_ylabel("V̄(z), electrostatic (eV)")
    axis.set_title(f"{name}: ΔV̄ = {potential_lineup[name]:.3f} eV, ΔE_v = {valence_band_offset[name]:.3f} eV")
    axis.legend(loc="lower right", bbox_to_anchor=(1, 0.07), fontsize=8)
figure.tight_layout()
display_matplotlib_figure(figure)

## 11. Compare with the manuscripts

Both publications define the offset as E_VBM(SiO₂) − E_VBM(HfO₂). Robertson and Wallace read it from the densities of
states of layers away from the interfaces; here both offsets come from the potential line-up, as in Prodhomme et al.
Settings that differ from the manuscripts: GBRV ultrasoft LDA pseudopotentials in plane waves at 40/200 Ry, against
the localized orbitals and Troullier–Martins pseudopotentials of Prodhomme et al.; the lattice constants of the platform's SiO₂ and HfO₂ structures, against the publications' LDA
lattices, so the in-plane strain of monoclinic HfO₂ is smaller than the 3.7 % and 2.1 % of Prodhomme et al.; and
the k-points of the bulks, which Prodhomme et al. do not give and Sharia et al. set to 8 × 8 × 8. The value of Robertson and Wallace is read from their Fig. 57a.

In [ ]:
MANUSCRIPT_OFFSET_BY_SUPERLATTICE_NAME = {
    MONOCLINIC_SUPERLATTICE_NAME: ("Prodhomme 2011", 0.4),  # eV, LDA, Table I
    CUBIC_SUPERLATTICE_NAME: ("Robertson and Wallace 2015", -0.67),  # eV, LDA, Fig. 57a
}

for name, (source, manuscript_offset) in MANUSCRIPT_OFFSET_BY_SUPERLATTICE_NAME.items():
    film_label = BLOCK_LABEL[FILM_BULK_NAME_BY_SUPERLATTICE_NAME[name]]
    print(f"VBO, SiO₂/HfO₂ ({film_label}): {source} {manuscript_offset:+} eV | ours {valence_band_offset[name]:+.2f} eV")

## References

[1] P.-Y. Prodhomme, F. Fontaine-Vive, A. Van Der Geest, P. Blaise and J. Even, "Ab initio calculation of effective
work functions for a TiN/HfO2/SiO2/Si transistor stack", Appl. Phys. Lett. 99, 022101 (2011).
[DOI: 10.1063/1.3609869](https://doi.org/10.1063/1.3609869)

[2] J. Robertson and R. M. Wallace, "High-K materials and metal gates for CMOS applications", Mater. Sci. Eng. R 88,
1 (2015). [DOI: 10.1016/j.mser.2014.11.001](https://doi.org/10.1016/j.mser.2014.11.001)

[3] O. Sharia, A. A. Demkov, G. Bersuker and B. H. Lee, "Theoretical study of the insulator/insulator interface: Band
alignment at the SiO2/HfO2 junction", Phys. Rev. B 75, 035306 (2007).
[DOI: 10.1103/PhysRevB.75.035306](https://doi.org/10.1103/PhysRevB.75.035306)

[4] K. F. Garrity, J. W. Bennett, K. M. Rabe and D. Vanderbilt, "Pseudopotentials for high-throughput DFT
calculations", Comput. Mater. Sci. 81, 446 (2014). https://doi.org/10.1016/j.commatsci.2013.08.053

[5] MACE-MP-0 foundation models: https://github.com/ACEsuit/mace